In [1]:
import torch
import torch.nn as nn
import os
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import torch.nn.functional as F
# from efficientnet_pytorch import EfficientNet
import torchvision.models as models
import pandas as pd
from tqdm import tqdm

import torch.optim as optim
from torch.utils.data import DataLoader

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
class MyImageFolder(Dataset):
    def __init__(self, root_dir, transform=None):
        super(MyImageFolder, self).__init__()
        self.data = []
        self.root_dir = root_dir
        self.transform = transform
        self.class_names = os.listdir(root_dir)

        for index, name in enumerate(self.class_names):
            files = os.listdir(os.path.join(root_dir, name))
            self.data += list(zip(files, [index]*len(files)))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_file, label = self.data[index]
        root_and_dir = os.path.join(self.root_dir, self.class_names[label])
        image = np.array(Image.open(os.path.join(root_and_dir, img_file)))

        if self.transform is not None:
            augmentations = self.transform(image=image)
            image = augmentations["image"]

        return image, label

In [4]:
models.inception_v3(pretrained=True)

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


  0%|          | 0.00/104M [00:00<?, ?B/s]

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [5]:
class Net(nn.Module):
    def __init__(self, net_version, num_classes):
        super(Net, self).__init__()
        # self.backbone = EfficientNet.from_pretrained('efficientnet-'+net_version)
        self.backbone = models.inception_v3(pretrained=True)
        self.backbone._fc = nn.Sequential(
            nn.Linear(2048, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)

In [6]:
def check_accuracy(loader, model, device="cuda"):
    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)

            scores = torch.sigmoid(model(x))
            predictions = (scores>0.5).float()
            num_correct += (predictions == y).sum()
            num_samples += predictions.shape[0]

        print(f'Got {num_correct} / {num_samples} with accuracy {float(num_correct) / float(num_samples) * 100:.2f}')

    model.train()


def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)


def load_checkpoint(checkpoint, model, optimizer):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint['state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer'])

def make_prediction(model, transform, rootdir, device):
    files = os.listdir(rootdir)
    preds = []
    model.eval()

    files = sorted(files, key=lambda x: float(x.split(".")[0]))
    for file in tqdm(files):
        img = Image.open(os.path.join(rootdir, file))
        img = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = torch.sigmoid(model(img))
            preds.append(pred.item())


    df = pd.DataFrame({'id': np.arange(1, len(preds)+1), 'label': np.array(preds)})
    df.to_csv('submission.csv', index=False)
    model.train()
    print("Done with predictions")

In [7]:
!pip install albumentations

     |████████████████████████████████| 631 kB 9.1 MB/s 
  Created wheel for imgaug: filename=imgaug-0.2.6-py3-none-any.whl size=654020 sha256=d3f22db0879c20213aed2ed5aafc9b11396a09ee0fc60964e2388ce1525cdece
  Stored in directory: /root/.cache/pip/wheels/89/72/98/3ebfdba1069a9a8eaaa7ae7265cfd67d63ef0197aaee2e5f9c
Successfully built imgaug
  Attempting uninstall: imgaug
    Found existing installation: imgaug 0.2.9
    Uninstalling imgaug-0.2.9:
      Successfully uninstalled imgaug-0.2.9


In [8]:
import albumentations as A
from albumentations.pytorch import ToTensor

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
BATCH_SIZE = 64
PIN_MEMORY = True
LOAD_MODEL = True
LEARNING_RATE = 1e-4
NUM_EPOCHS = 100

train_transforms = A.Compose([
    A.Resize(width=224, height=224,),
    A.RandomCrop(width=224, height=224),
    A.Rotate(40),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.Normalize(
        mean=[0, 0, 0],
        std=[1, 1, 1],
        max_pixel_value=255.0,
    ),
    ToTensor,
])

val_transforms = A.Compose([
    A.Resize(height=224, width=224),
    A.Normalize(
        mean=[0, 0, 0],
        std=[1, 1, 1],
        max_pixel_value=255.0,
    ),
    ToTensor(),
])

In [9]:
def train_fn(loader, model, optimizer, loss_fn, scaler, device):
    for batch_idx, (data, targets) in enumerate(tqdm(loader)):
        # Get data to cuda if possible
        data = data.to(device=device)
        targets = targets.to(device=device)

        # forward
        with torch.cuda.amp.autocast():
            scores = model(data)
            loss = loss_fn(scores, targets.float())

        # backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

In [14]:
train_ds = MyImageFolder(root_dir="/content/gdrive/MyDrive/datasets/Cats&Dogs/train", transform=train_transforms)
val_ds = MyImageFolder(root_dir="/content/gdrive/MyDrive/datasets/Cats&Dogs/valid", transform=val_transforms)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY,shuffle=True)

loss_fn = nn.CrossEntropyLoss()
model = Net(net_version="b0", num_classes=10).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler()

# if LOAD_MODEL:
#     load_checkpoint(torch.load("my_checkpoint.pth.tar"), model, optimizer)

# make_prediction(model, val_transforms, '/content/gdrive/MyDrive/datasets/dataset_basura/', DEVICE)
# check_accuracy(val_loader, model, DEVICE)

for epoch in range(NUM_EPOCHS):
    train_fn(train_loader, model, optimizer, loss_fn, scaler, DEVICE)
    check_accuracy(val_loader, model, DEVICE)
    checkpoint = {'state_dict': model.state_dict(), 'optimizer': optimizer.state_dict()}
    save_checkpoint(checkpoint)


  0%|          | 0/360 [00:00<?, ?it/s]


TypeError: ignored

In [ ]:
train_ds = MyImageFolder(root_dir="/content/gdrive/MyDrive/datasets/Cats&Dogs/train", transform=train_transforms)
val_ds = MyImageFolder(root_dir="/content/gdrive/MyDrive/datasets/Cats&Dogs/valid", transform=val_transforms)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY,shuffle=True)

loss_fn = nn.CrossEntropyLoss()
model = Net(net_version="b0", num_classes=10).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler()

# if LOAD_MODEL:
#     load_checkpoint(torch.load("my_checkpoint.pth.tar"), model, optimizer)

make_prediction(model, val_transforms, '/content/gdrive/MyDrive/datasets/dataset_basura/', DEVICE)
check_accuracy(val_loader, model, DEVICE)

for epoch in range(NUM_EPOCHS):
    train_fn(train_loader, model, optimizer, loss_fn, scaler, DEVICE)
    check_accuracy(val_loader, model, DEVICE)
    checkpoint = {'state_dict': model.state_dict(), 'optimizer': optimizer.state_dict()}
    save_checkpoint(checkpoint)


ValueError: ignored

In [12]:
train_ds = MyImageFolder(root_dir="/content/gdrive/MyDrive/datasets/Cats&Dogs/train", transform=train_transforms)
val_ds = MyImageFolder(root_dir="/content/gdrive/MyDrive/datasets/Cats&Dogs/valid", transform=val_transforms)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,pin_memory=PIN_MEMORY,shuffle=True)

loss_fn = nn.CrossEntropyLoss()
model = Net(net_version="b0", num_classes=2).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler()

# if LOAD_MODEL:
#     load_checkpoint(torch.load("my_checkpoint.pth.tar"), model, optimizer)

make_prediction(model, val_transforms, '/content/gdrive/MyDrive/datasets/Cats&Dogs/test', DEVICE)
check_accuracy(val_loader, model, DEVICE)

for epoch in range(NUM_EPOCHS):
    train_fn(train_loader, model, optimizer, loss_fn, scaler, DEVICE)
    check_accuracy(val_loader, model, DEVICE)
    checkpoint = {'state_dict': model.state_dict(), 'optimizer': optimizer.state_dict()}
    save_checkpoint(checkpoint)


  0%|          | 0/12500 [00:00<?, ?it/s]


TypeError: ignored